### In my last notebook, MFSGD model performed really bad on model evaluation. I will conduct Optuna Study to find the best hyperparameters to see if the scores improved or not.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import optuna
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/movies.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/README.txt
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/tags.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/links.csv


<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">Load the DataFrame</span> 
  </h1>
</div>

In [42]:
df = pd.read_csv('/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv')

In [43]:
# First five rows
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [44]:
df = df.rename(columns = {'userId': 'user_id', 'movieId': 'movie_id'})

In [ ]:
df.head()

In [45]:
df.shape

(100836, 4)

In [46]:
df.describe()

,user_id,movie_id,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [47]:
df['user_id'] = df['user_id'] - 1
df['movie_id'] = df['movie_id'] - 1


In [48]:
df.describe()

,user_id,movie_id,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,325.127564,19434.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,0.000000,0.000000,0.500000,8.281246e+08
25%,176.000000,1198.000000,3.000000,1.019124e+09
50%,324.000000,2990.000000,3.500000,1.186087e+09
75%,476.000000,8121.000000,4.000000,1.435994e+09
max,609.000000,193608.000000,5.000000,1.537799e+09


<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">MatrixFactorizationSGD</span> 
  </h1>
</div>

In [57]:
class MatrixFactorizationSGD:
    """
    Matrix Factorization with SGD — built directly on your SGDRegressor style
    Predicts rating = user_factor[u] dot item_factor[i]
    """
    def __init__(self, n_factors=20, learning_rate=0.01, epochs=50, 
                 batch_size=1, reg=None, reg_param=0.01):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        
        self.P = None  # user latent factors
        self.Q = None  # item latent factors
        self.global_bias = 0.0
        
    def fit(self, user_ids, item_ids, ratings):
        """
        user_ids, item_ids, ratings: numpy arrays (or pandas Series) of same length
        """
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        ratings = np.asarray(ratings, dtype=float)
        
        n_users = 610
        n_items = 193609
        
        
        # Initialize latent factors (small random values)
        self.P = np.random.normal(0, 0.01, (n_users, self.n_factors))
        self.Q = np.random.normal(0, 0.01, (n_items, self.n_factors))
        self.global_bias = np.mean(ratings)
        
        for epoch in range(self.epochs):
            # Shuffle (stochastic part)
            indices = np.random.permutation(len(ratings))
            for i in range(0, len(ratings), self.batch_size):
                batch_index = indices[i:i + self.batch_size]
                u_batch = user_ids[batch_index]
                i_batch = item_ids[batch_index]
                r_batch = ratings[batch_index]
                
                # Forward pass
                pred = np.sum(self.P[u_batch] * self.Q[i_batch], axis=1) + self.global_bias
                error = r_batch - pred
                
                # Gradients (same style as your SGDRegressor!)
                grad_P = -2 * (error[:, np.newaxis] * self.Q[i_batch])
                grad_Q = -2 * (error[:, np.newaxis] * self.P[u_batch])
                
                # Regularization (exactly like your code)
                if self.reg == 'l2':
                    grad_P += 2 * self.reg_param * self.P[u_batch]
                    grad_Q += 2 * self.reg_param * self.Q[i_batch]
                elif self.reg == 'l1':
                    grad_P += self.reg_param * np.sign(self.P[u_batch])
                    grad_Q += self.reg_param * np.sign(self.Q[i_batch])
                
                # SGD update
                self.P[u_batch] -= self.learning_rate * grad_P
                self.Q[i_batch] -= self.learning_rate * grad_Q
        
        return self
    
    def predict(self, user_ids, item_ids):
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        
        return np.sum(self.P[user_ids] * self.Q[item_ids], axis=1) + self.global_bias
    

<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">Splitting the Data into Train and Test</span> 
  </h1>
</div>

In [58]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=32, stratify=df['user_id']) 

# Our model expect arrays 
train_users = train_df['user_id'].values
train_movies = train_df['movie_id'].values
train_ratings = train_df['rating'].values

test_users = test_df['user_id'].values
test_movies = test_df['movie_id'].values
test_ratings = test_df['rating'].values


In [55]:
model = MatrixFactorizationSGD(n_factors=10, learning_rate=0.01, epochs=50, 
                               reg='l1', reg_param=0.02, batch_size=32)

model.fit(train_users, train_movies, train_ratings)

<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">Optuna Study</span> 
  </h1>
</div>

### Conducting optuna study for find the optima hyperparameter configurations for our model:
- setting parameters
- use MatrixFactorizationSGD as the predictive model


In [67]:
def objective(trial):

    reg_choice = trial.suggest_categorical('reg', ['l1', 'l2'])
    reg = reg_choice
    reg_param = trial.suggest_float('reg_param', 1e-5, 10.0, log=True)
        
    params = {
        'n_factors': trial.suggest_int('n_factors', 5, 150, step=5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.05, log=True),
        'epochs': trial.suggest_int('epochs', 5, 200),
        'batch_size': trial.suggest_int('batch_size', 16, 320, step=16),  
        'reg': reg,
        'reg_param': reg_param,
    }
    
    model = MatrixFactorizationSGD(**params).fit(train_users, train_movies, train_ratings)

    pred = model.predict(test_users, test_movies)
    rmse = np.sqrt(mean_squared_error(test_ratings, pred))
    return rmse

study = optuna.create_study(direction='minimize', study_name='MFSGD-Optimization')
study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2026-03-14 09:45:46,262] A new study created in memory with name: MFSGD-Optimization


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-03-14 09:46:03,640] Trial 0 finished with value: 1.0507002018575902 and parameters: {'reg': 'l1', 'reg_param': 0.009370174734042823, 'n_factors': 80, 'learning_rate': 0.012139271357882167, 'epochs': 119, 'batch_size': 272}. Best is trial 0 with value: 1.0507002018575902.
[I 2026-03-14 09:46:25,184] Trial 1 finished with value: 1.0465884715591205 and parameters: {'reg': 'l1', 'reg_param': 0.12104590351444772, 'n_factors': 40, 'learning_rate': 0.032844663042691506, 'epochs': 110, 'batch_size': 32}. Best is trial 1 with value: 1.0465884715591205.
[I 2026-03-14 09:46:39,835] Trial 2 finished with value: 1.0465856075227316 and parameters: {'reg': 'l2', 'reg_param': 0.4809036718553459, 'n_factors': 110, 'learning_rate': 0.0005814418933132077, 'epochs': 89, 'batch_size': 144}. Best is trial 2 with value: 1.0465856075227316.
[I 2026-03-14 09:46:52,999] Trial 3 finished with value: 1.0500612238873368 and parameters: {'reg': 'l2', 'reg_param': 3.311456873450563e-05, 'n_factors': 45, 'lea

In [68]:
print("\n✅ BEST HYPERPARAMETERS FOUND:")
print(study.best_params)
print(f"✅ Best RMSE: {study.best_value:.4f}")


✅ BEST HYPERPARAMETERS FOUND:
{'reg': 'l2', 'reg_param': 0.03668745633333413, 'n_factors': 135, 'learning_rate': 0.0011932713389009044, 'epochs': 135, 'batch_size': 80}
✅ Best RMSE: 0.8864


In [69]:
best_params = study.best_params.copy()
best_model = MatrixFactorizationSGD(**best_params).fit(train_users, train_movie, train_rating)
final_pred = best_model.predict(test_users, test_movie)

In [70]:
rmse = np.sqrt(mean_squared_error(test_ratings, final_pred))
mae = mean_absolute_error(test_ratings, final_pred)
r2score = r2_score(test_ratings, final_pred)
explained_var = explained_variance_score(test_ratings, final_pred)

print(f"Final Validation RMSE: {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2score:.4f}")
print(f'Explained Variance Score: {explained_var:.4f}')

Final Validation RMSE: 0.8866
Mean Absolute Error (MAE): 0.6787
R² Score: 0.2823
Explained Variance Score: 0.2851


#### As you can see that scores have improved but r2score and explained variance score are still low. I am not content with scores. Let's see if implementing KFold would help to improve the scores.